# Two-arm MMIDAS on snRNA (Dbh), across 5 random seeds

**Pipeline order** (run from the repo root):
1. `notebooks/00a_prepare_data_for_mmidas.ipynb` — build `data/snRNA_BN_norm1.h5ad`.
2. `sbatch scripts/train_aug_mixvae.sh` — train the augmenter. It writes `models/RNA_augmenter_Dbh_<timestamp>.pth`; set `augmenter_Dbh` in `pyproject.toml` to that filename (or `"."` to force fresh training).
3. `sbatch scripts/train_mixvae_slurmarray.sh` — train the 2-arm model for seeds 1–5.
4. This notebook — load the trained models, select K, and make the summary figures.

**Data availability:** the source data is not tracked in git. Download `snRNAseq_LCNE_BN_d4_1-5k.h5ad` from Code Ocean and place it under `data/`, then run `00a`:
https://codeocean.allenneuraldynamics.org/data-assets/915f30e7-324d-4ed5-a520-5509a6e54f87/LCNE-transcriptomics-preprocessing_2026-07-08_16-23-00

The augmenter `.pth`, trained models, and result `.pkl` files are produced by the training scripts (steps 2–3); place any pre-trained/deposited copies under `models/` and the `results/` path defined in `pyproject.toml`.

# Load the multiseed results

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, glob
import torch
print(sys.executable) 
print(torch.cuda.is_available())

In [ ]:
import scanpy as sc
import anndata as ad
import umap
import numpy as np
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt

from mmidas.vaegan import vae_gan
from mmidas.utils.config_tools import get_paths
from mmidas.utils.data_tools import load_data_BN, load_data_raw,  get_loaders, generate_colors
from mmidas.utils.augmentation import get_loader

from mmidas.cplMixVAE import cpl_mixVAE
from mmidas.eval import summarize_inference
from mmidas.utils.cluster_analysis import K_selection
from mpl_toolkits.axes_grid1 import make_axes_locatable
import seaborn as sns


In [ ]:
toml_file = 'pyproject.toml'
device = 0  # GPU index passed to cpl_mixVAE
config = get_paths(toml_file=toml_file)
aug_path = config['paths']['main_dir'] / config['paths']['models']
aug_path

# load data

In [ ]:
data = load_data_BN()
data_loader = get_loader(x=data['log1p'] , batch_size=256, training=False)

# load the results

In [ ]:

results_path = config['paths']['main_dir'] / config['paths']['saving_path_randomseed']
results_path


In [ ]:
# random seeds whose trained-model folders we load (set by train_mixvae_slurmarray.sh)
myinterestedseed = [1, 2, 3, 4, 5]
available_models_all = glob.glob(str(results_path) + '/randomseed*')
available_models = [s for s in available_models_all if any(f"randomseed_{i}_" in s for i in myinterestedseed)]
len(available_models)

In [ ]:
def get_model_params(selected_model_file):
    param_variables = selected_model_file.split('/')[-1].split('_')
    params = {}
    for p in range(0, len(param_variables), 2):
        try:
            params[param_variables[p]] = float(param_variables[p+1])
            if params[param_variables[p]] == int(params[param_variables[p]]):
                params[param_variables[p]] = int(params[param_variables[p]])
        except ValueError:
            params[param_variables[p]] = param_variables[p+1]
    return params

def print_dates_model_info(trained_models_ALLDATE, all_time_stamps_available):
    '''Report, per timestamp, how many model files are available.'''
    from pprint import pprint
    for m in all_time_stamps_available:
        print(m, len([modelname for modelname in trained_models_ALLDATE if m in modelname]))
        pprint([modelname.split('/')[-1].split('cpl_mixVAE_model_after_pruning_')[-1]
                for modelname in trained_models_ALLDATE if m in modelname])

def get_trained_models(trained_models_ALLDATE, all_time_stamps_available):
    '''Pick the timestamp with the most model files; if several tie, use the most recent one.'''
    counts = {m: sum(m in modelname for modelname in trained_models_ALLDATE)
              for m in all_time_stamps_available}
    max_count = max(counts.values())
    best_m_list = [m for m, c in counts.items() if c == max_count]
    best_m = sorted(best_m_list)[-1]  # timestamps sort chronologically -> most recent
    print('***** using model from this date:', best_m)
    trained_models = [models for models in trained_models_ALLDATE if best_m in models]
    return trained_models

In [ ]:
all_results = []
all_K_values = []
for i in range(len(myinterestedseed)):
    print(f"\n*** seed {myinterestedseed[i]} (model set {i}) ")
    selected_model_file = available_models[i]
    trained_models_ALLDATE = glob.glob(selected_model_file + '/model/cpl_mixVAE_model_*')
    splitter = 'cpl_mixVAE_model_before_pruning_'
    all_time_stamps_available = [m.split('/')[-1].split(splitter)[-1].split('.pth')[0]
                                 for m in trained_models_ALLDATE if splitter in m]
    trained_models = get_trained_models(trained_models_ALLDATE, all_time_stamps_available)
    params = get_model_params(selected_model_file)

    print('Initializing mixvae model ...')
    mixvae = cpl_mixVAE(saving_folder=selected_model_file, device=device)
    mixvae.init_model(
                    n_categories=params['Cdim'],
                    state_dim=params['Sdim'],
                    input_dim=data['log1p'].shape[1],
                    fc_dim=params['fcdim'],
                    lowD_dim=params['Zdim'],
                    n_arm=params['narm'],
                    tau=params['tau'],
                    )
    alldata_loader, train_loader, test_loader, _, _, _ = get_loaders(x=data['log1p'], batch_size=params['nbatch'],
                                                                      seed=params['randomseed'])
    mixvae.variational = False
    summary_dict = summarize_inference(mixvae, trained_models, test_loader)
    num_pruned, l_recon_mean, consensus, K = K_selection(summary_dict, mixvae.n_categories, mixvae.n_arm,
                                                         thr=0.99, plot_reconst=True)
    all_results.append({
        'num_pruned': num_pruned,
        'l_recon_mean': l_recon_mean,
        'consensus': consensus,
        'summary_dict': summary_dict
    })
    all_K_values.append(K)
    print(f"Seed {myinterestedseed[i]}: Selected K = {K}")

In [ ]:
# Now compute the mean across all seeds
print(f"\nComputing mean across {len(myinterestedseed)} seeds...")
common_num_pruned = all_results[0]['num_pruned']
all_l_recon = []
all_consensus = []
all_d_qc = []

for result in all_results:
    assert np.array_equal(result['num_pruned'], common_num_pruned), "num_pruned should be the same across seeds"    
    all_l_recon.append(result['l_recon_mean'])
    all_consensus.append(result['consensus'])
    indx = np.argsort(result['summary_dict']['num_pruned'])
    all_d_qc.append(np.array(result['summary_dict']['d_qc'])[indx])

# Compute means and standard deviations
mean_l_recon = np.mean(all_l_recon, axis=0)
std_l_recon = np.std(all_l_recon, axis=0)

mean_consensus = np.mean(all_consensus, axis=0)
std_consensus = np.std(all_consensus, axis=0)

mean_d_qc = np.mean(all_d_qc, axis=0)
std_d_qc = np.std(all_d_qc, axis=0)

mean_dissent = 1 - mean_consensus
std_dissent = std_consensus  # std of (1-x) = std of x

# Normalize reconstruction loss for plotting
norm_l_recon = mean_l_recon/ data['log1p'].shape[1]
norm_std_l_recon = std_l_recon / data['log1p'].shape[1]


In [ ]:


# Plot the averaged results
fig = plt.figure(figsize=[8, 6])
ax = fig.add_subplot()
ax.errorbar(common_num_pruned, mean_d_qc, yerr=std_d_qc, label='Average Distance', capsize=3)
ax.errorbar(common_num_pruned, mean_dissent, yerr=std_dissent, label='Average Dissent (1 - Consensus)', capsize=3)
ax.errorbar(common_num_pruned, norm_l_recon,yerr=norm_std_l_recon,
           label='per feature recon loss', capsize=3)

ax.set_xlim([np.min(common_num_pruned)-1, mixvae.n_categories + 1])
ax.set_xlabel('Categories', fontsize=14)
ax.set_xticks(common_num_pruned)
ax.set_xticklabels(common_num_pruned, fontsize=8, rotation=90)
ax.set_title(f'Mean across {len(myinterestedseed)} seeds - Seeking consensus {mixvae.n_arm}-arm MMIDAS', fontsize=14)

ax.legend(loc='upper right')
plt.show()

print(f"Selected K values across seeds: {all_K_values}")

In [ ]:
fig = plt.figure(figsize=[10, 6])
ax = fig.add_subplot()

# Plot lines with shaded bands instead of error bars
ax.plot(common_num_pruned, mean_d_qc, label='Average Distance', marker='x',linewidth=2)
ax.fill_between(common_num_pruned, mean_d_qc - std_d_qc, mean_d_qc + std_d_qc, alpha=0.3)

ax.plot(common_num_pruned, mean_dissent, label='Average Dissent (1 - Consensus)', marker='x',linewidth=2)
ax.fill_between(common_num_pruned, mean_dissent - std_dissent, mean_dissent + std_dissent, alpha=0.3)

ax.plot(common_num_pruned, norm_l_recon, label='Per Feature Recon Loss', marker='x',linewidth=2)
ax.fill_between(common_num_pruned, norm_l_recon - norm_std_l_recon, norm_l_recon + norm_std_l_recon, alpha=0.3)

ax.set_xlim([np.min(common_num_pruned)-1, mixvae.n_categories + 1])
ax.set_xlabel('Categories', fontsize=14)
ax.set_xticks(common_num_pruned)
ax.set_xticklabels(common_num_pruned, fontsize=8, rotation=90)
ax.set_title(f'Mean across {len(myinterestedseed)} seeds - Seeking consensus {mixvae.n_arm}-arm MMIDAS', fontsize=14)

ax.legend(loc='upper right')
plt.show()

print(f"Selected K values across seeds: {all_K_values}")



# Save per-model inference results for downstream silhouette-score analysis

In [ ]:
all_results = []

import pickle as pkl
for i in range(len(myinterestedseed)):
    print(f"\n*** seed {myinterestedseed[i]} (model set {i}) ")
    selected_model_file = available_models[i]  # the random seed only selects which trained-model set to load
    trained_models_ALLDATE = glob.glob(selected_model_file + '/model/cpl_mixVAE_model_*')
    splitter = 'cpl_mixVAE_model_before_pruning_'
    all_time_stamps_available = [m.split('/')[-1].split(splitter)[-1].split('.pth')[0]
                                 for m in trained_models_ALLDATE if splitter in m]
    trained_models = get_trained_models(trained_models_ALLDATE, all_time_stamps_available)
    params = get_model_params(selected_model_file)
    mixvae = cpl_mixVAE(saving_folder=selected_model_file, device=device)
    mixvae.init_model(n_categories=params['Cdim'], state_dim=params['Sdim'],
                    input_dim=data['log1p'].shape[1], fc_dim=params['fcdim'],
                    lowD_dim=params['Zdim'], n_arm=params['narm'], tau=params['tau'],)
    alldata_loader, train_loader, test_loader, _, _, _ = get_loaders(x=data['log1p'], batch_size=params['nbatch'],
                                                                      seed=params['randomseed'])
    mixvae.variational = False
    for file in trained_models:
        file_name_ind = file.rfind('/')
        print(f'Model {file[file_name_ind:]}')
        all_results.append({"seed_i": i, "model_file": file, "outcome": summarize_inference(mixvae, file, alldata_loader)})

# saved for downstream silhouette-score analysis (see Data availability)
mmidas_res_path = str(config['paths']['main_dir'] / config['paths']['saving_path'] / 'all_mmidas_outcome_5seed.pkl')
pkl.dump(all_results, file=open(mmidas_res_path, 'wb'))

In [ ]:
len(np.unique([r['seed_i'] for r in all_results])), len(np.unique([r['outcome']['num_pruned'][0] for r in all_results]))